# 1. Project and dataset introduction

This notebook describes the single immutable root dataset, `nft.csv`, using the same audited and cleaned contracts as the command-line modules. Despite the filename, the observed schema supports DEX/swap-event semantics, not verified NFT-sale semantics. No wash-trading label is available, so all findings are descriptive rather than evidence of misconduct.

## 2. Research objectives, data limitations, and reproducible setup

The objective is to understand validated DEX/swap activity and its suitability for later point-in-time wallet risk research. The source lacks verified wash-trading labels, buyer/seller roles, counterparty edges, documented provenance, and broad longitudinal coverage, so this notebook makes no supervised, graph, causal, or NFT-marketplace claim. Repository-relative paths replace the lecturer notebook's machine-specific path. Plot settings are fixed and direct wallet, transaction, and token identifiers are not displayed.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = next(path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (path / 'pyproject.toml').is_file())
sys.path.insert(0, str(ROOT / 'src'))
os.environ.setdefault('MPLCONFIGDIR', '/tmp/solana-risk-matplotlib')

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from solana_risk.data.audit import audit_dataset
from solana_risk.data.preprocess import preprocess_dataset

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 80)
RAW_PATH = ROOT / 'nft.csv'
EXPECTED_SHA256 = '8f387709c6742f2fc31c19fc32dac27474b6b27b9304786c367a8d8bece81271'

## 3. Dataset loading and schema inspection

In [ ]:
audit = audit_dataset(RAW_PATH)
assert audit['source']['sha256'] == EXPECTED_SHA256
audit['source']

## 4. Validated Phase 2 event loading

In [ ]:
preprocessing = preprocess_dataset(RAW_PATH, expected_sha256=EXPECTED_SHA256)
events = preprocessing.cleaned.copy()
events['block_time'] = pd.to_datetime(events['block_time'], utc=True)
{'retained_rows': len(events), 'quarantined_rows': len(preprocessing.quarantine), 'columns': len(events.columns)}

## 5. Dataset shape, column types, and event grain

In [ ]:
display(events.dtypes.astype(str).rename('dtype').to_frame())
display(events.drop(columns=['wallet', 'tx_hash', 'token_in', 'token_out', 'raw__wallet', 'raw__tx_hash', 'raw__token_in', 'raw__token_out'], errors='ignore').head(3))

## 6. Missing-value analysis

The bar chart measures prevalence and the heatmap shows co-occurrence without claiming a missingness mechanism. Fee absence dominates and must remain distinct from a valid zero.

In [ ]:
missing = events[['fee_usd', 'token_pair', 'outer_instruction_index', 'inner_instruction_index']].isna()
display(missing.sum().rename('missing_count').to_frame())
missing.mean().sort_values().plot.barh(title='Missing-value proportion in retained events', xlabel='proportion');
plt.tight_layout(); plt.show()
sns.heatmap(missing.head(250).T, cbar=False, yticklabels=True, xticklabels=False);
plt.title('Missingness indicators: first 250 ordered events'); plt.show()

## 7. Duplicate and multi-instruction evidence

In [ ]:
tx_sizes = events.groupby('tx_hash', sort=False).size()
pd.Series({'exact_cleaned_rows': int(events.duplicated().sum()), 'repeated_transaction_hashes': int(tx_sizes.gt(1).sum()), 'maximum_events_per_transaction': int(tx_sizes.max())})

## 8. Transaction amount distributions

Log axes make the strongly right-skewed positive amounts readable. The long upper tail motivates robust later transformations inside training folds, not deletion during EDA.

In [ ]:
display(events['amount_usd'].describe(percentiles=[.5, .9, .95, .99]).to_frame())
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(events['amount_usd'], bins=50, log_scale=(True, True), ax=axes[0]); axes[0].set_title('USD event amount (log-log)')
sns.boxplot(x=events['amount_usd'], log_scale=True, ax=axes[1]); axes[1].set_title('USD event amount (log scale)')
plt.tight_layout(); plt.show()

## 9. Wallet activity distributions

The log-log histogram makes the large mass of low-activity wallets and small high-activity tail visible; this uneven evidence limits comparisons between wallets.

In [ ]:
wallet_counts = events.groupby('wallet').size()
display(wallet_counts.describe(percentiles=[.5, .9, .95, .99]).rename('events_per_wallet').to_frame())
sns.histplot(wallet_counts, bins=40, log_scale=(True, True)); plt.title('Events per wallet (log-log)'); plt.show()

## 10. DEX distributions

The top-label chart describes routing concentration after formatting-only normalization. Protocol names are not merged into invented families.

In [ ]:
dex_counts = events['dex'].value_counts().head(15).sort_values()
dex_counts.plot.barh(title='Top 15 formatting-normalized DEX labels', xlabel='events'); plt.tight_layout(); plt.show()

## 11. Token distributions without exposing identifiers

Per-wallet diversity is summarized without listing token mints. Most wallets trade few distinct tokens, so diversity and concentration must be interpreted alongside event count.

In [ ]:
wallet_token_counts = events.groupby('wallet').apply(lambda frame: pd.concat([frame['token_in'], frame['token_out']]).nunique(), include_groups=False)
display(wallet_token_counts.describe(percentiles=[.5, .9, .95, .99]).rename('unique_tokens_per_wallet').to_frame())
sns.histplot(wallet_token_counts, discrete=True); plt.title('Unique token count per wallet'); plt.show()

## 12. Temporal activity trends and longitudinal limits

The monthly line verifies coverage through November 2025. Wallet-month counts show that 2,302 of 2,312 wallets occur in only one month, making cross-period behavioral claims especially weak.

In [ ]:
monthly = events.set_index('block_time').resample('MS').size()
monthly.plot(marker='o', title='Validated event count by UTC month', ylabel='events'); plt.show()
wallet_months = events.assign(month=events['block_time'].dt.to_period('M')).groupby('wallet')['month'].nunique()
display(wallet_months.value_counts().sort_index().rename_axis('observed_months').rename('wallet_count').to_frame())

## 13. Outlier investigation is descriptive, not deletion

The IQR rule is used only to locate values for review and compare their DEX-level summaries. It is not a correctness test or a wash-trading label.

In [ ]:
q1, q3 = events['amount_usd'].quantile([.25, .75])
upper = q3 + 1.5 * (q3 - q1)
extreme = events.loc[events['amount_usd'].gt(upper), ['amount_usd', 'dex', 'block_time']]
display(pd.Series({'iqr_review_threshold_usd': upper, 'flagged_events': len(extreme), 'flagged_share': len(extreme) / len(events)}))
display(extreme.groupby('dex')['amount_usd'].agg(['count', 'median', 'max']).sort_values('count', ascending=False).head(10))

## 14. Observations about class-label availability

No verified wash-trading target, buyer/seller pair, or wallet-counterparty edge is present.

## 15. Data quality and leakage risks

Wallet, transaction, and token identifiers are metadata or aggregation keys—not direct model inputs. Supplied `prev_event_time`, `seconds_since_prev`, and `source_week` are not trusted because their lineage is unverified; Phase 3 recomputes time history from validated `block_time`. Missing fee means unavailable, not zero.

## 16. Key findings and implications for modelling

Heavy-tailed amounts remain review flags rather than fabricated errors. Shallow wallet history is the dominant limitation: descriptive point-in-time behavior can be engineered, but supervised and graph claims remain blocked. Later fitted preprocessing must remain inside chronological folds, after verified labels become available.

In [ ]:
target_like = {'label', 'target', 'is_wash_trade', 'wash_trade', 'is_fraud', 'fraud_label', 'risk_label'}
assert target_like.isdisjoint({column.casefold() for column in events.columns})
{'verified_label_available': False, 'wallet_counterparty_edge_available': False, 'safe_next_step': 'point-in-time descriptive wallet features only'}